In [ ]:
# Hugging Face 提供了多种自动模型加载类。
#
# 它们会根据模型配置选择适当的模型实现。
# API	主要用途	典型输出
# AutoModel	加载基础模型主体	Hidden States
# AutoModelForSequenceClassification	文本分类	分类 Logits
# AutoModelForCausalLM	自回归语言模型	词表 Logits

In [ ]:
import random
from typing import Any

from torch.utils.data.dataset import _T_co
# AutoModel
from transformers import AutoModel

model = AutoModel.from_pretrained('distilbert/distilbert-base-uncased')

In [ ]:
# AutoModelForSequenceClassification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

# 它加载适合序列分类的模型结构，包括分类头。

In [ ]:
# AutoModelForCausalLM

from transformers import AutoModelForCausalLM

# 这是后续训练 GPT 类自回归语言模型时的重要接口。

In [ ]:
# inspect_transformer.py
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "distilbert/distilbert-base-uncased"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device:", device)

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 加载预训练模型
model = AutoModel.from_pretrained(MODEL_ID)

# 将模型移动到GPU
model = model.to(device)

# 切换评估模式
model.eval()

# 输入文本
texts = [
    "I love machine learning.",
    "Transformers are powerful models."
]

# 文本转Tensor
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

print("\nInput IDs:")
print(inputs["input_ids"])

print("\nAttention Mask:")
print(inputs["attention_mask"])

# 输出 Tensor 转移到 GPU
inputs = {
    key: value.to(device) for key, value in inputs.items()
}

# for key, value in inputs.items():
#     print(f"{key}: {value}")

# 执行模型前向传播
with torch.no_grad():
    outputs = model(**inputs)

# 查看输出
hidden_states = outputs.last_hidden_state
print("\nHidden States Shape:")
print(hidden_states.shape)

# 统计模型参数
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)

# 查看模型配置
print("\nModel configuration:")
print("Hidden size:", model.config.dim)
print("Layers:", model.config.n_layers)
print("Attention heads:", model.config.n_heads)

In [ ]:
# inspect_layers.py


import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)

model.eval()

inputs = tokenizer(
    "I am learning Transformer models.",
    return_tensors="pt",
)

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
    )

print("Number of hidden state tensors:")
print(len(outputs.hidden_states))

for index, hidden in enumerate(outputs.hidden_states):
    print(
        f"Hidden state {index}:",
        tuple(hidden.shape),
    )


In [ ]:
# Self Attention
import math
import torch

# 两个 Token 的输入向量
X = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0]
])

# 三个投影矩阵
W_Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0]
])

W_K = torch.tensor([
    [1.0, 1.0],
    [0.0, 1.0]
])

W_V = torch.tensor([
    [2.0, 0.0],
    [0.0, 3.0]
])

# 1. 生成 Q、K、V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# 2. 计算 Attention Scores
scores = Q @ K.T

# 3. 缩放
d_k = K.shape[-1]
scaled_scores = scores / math.sqrt(d_k)

# 4. Softmax
weights = torch.softmax(scaled_scores, dim=-1)

# 5. 加权求和
output = weights @ V

print("Q:\n", Q)
print("K:\n", K)
print("V:\n", V)
print("Scores:\n", scores)
print("Scaled Scores:\n", scaled_scores)
print("Weights:\n", weights)
print("Output:\n", output)

# Q @ K.T：计算匹配分数。
# / sqrt(d_k)：缩放分数。
# softmax()：计算注意力权重。
# weights @ V：根据权重汇总信息。

In [ ]:
# Transformer Block
import torch
from torch import nn

# 定义Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=4,
            num_heads=2,
            batch_first=True
        )

        # 第一次层归一化
        self.norm1 = nn.LayerNorm(4)

        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 4)
        )

        # 第二次层归一化
        self.norm2 = nn.LayerNorm(4)

    def forward(self, x):
        print("原始输入:", x.shape)

        seq_len = x.shape[1]

        causal_mask = torch.triu(
            torch.ones(
            seq_len,
            seq_len,
            device=x.device,
            dtype=torch.bool
            ),
            diagonal=1
        )

        # 第一部Attention
        attn_output, weights = self.attention(
            x, x, x,
            attn_mask=causal_mask
        )


        print("attn_output: ", attn_output)
        print("weights: ", weights)

        # 残差连接 + LayerNorm
        h = self.norm1(x + attn_output)
        print("h: ", h)

        # FNN
        ffn_output = self.ffn(h)
        print("ffn_output: ", ffn_output)

        # 残差连接 + LayerNorm
        output = self.norm2(h + ffn_output)
        print("output: ", output)

        return output

# 模拟三个token embedding
x = torch.tensor([[
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8],
    [0.9, 1.0, 1.1, 1.2]
]])


model = TransformerBlock()

output = model(x)

print("\n最终输出向量:")
print(output)


In [ ]:
torch.ones(3,3,device=x.device,dtype=torch.bool)

torch.triu(
            torch.ones(
            3,
            3,
            device=x.device,
            dtype=torch.bool
            ),
            diagonal=1
        )

In [ ]:
import torch
from torch import nn

# 定义词表
vocab = ["我", "喜欢", "苹果", "香蕉", "学习"]

# 2. 假设 Transformer 最后一个 Token 的输出
h = torch.tensor([[1.0, 2.0, 0.0, 1.0]])

print("隐藏向量形状:", h.shape)

# 3. 创建 LM Head
lm_head = nn.Linear(4, 5, bias=False)

# 4. 设置权重，方便和手算对照
with torch.no_grad():
    lm_head.weight.copy_(torch.tensor([
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
        [1.0, 2.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [1.0, 0.0, 0.0, 0.0]
    ]))


# 5. 计算 Logits
logits = lm_head(h)
print("Logits:", logits)

# 6. Softmax 转换成概率
probs = torch.softmax(logits, dim=-1)
print("概率:", probs)

# 7. 选择概率最大的 Token
next_token_id = torch.argmax(probs, dim=-1).item()

print("预测 Token ID:", next_token_id)
print("预测 Token:", vocab[next_token_id])

In [ ]:
import torch
from torch import nn

# 假设 Transformer 输出
hidden_states = torch.randn(2, 10, 768)

# 词表大小
vocab_size = 50000

# LM Head
lm_head = nn.Linear(768, vocab_size, bias=False)

# 计算 Logits
logits = lm_head(hidden_states)

print(logits.shape)
# torch.Size([2, 10, 50000])

# 获取每条序列最后一个位置的预测分数
next_token_logits = logits[:, -1, :]

print(next_token_logits.shape)
# torch.Size([2, 50000])

# 选择分数最高的 Token ID
next_token_id = next_token_logits.argmax(dim=-1)

print(next_token_id.shape)
# torch.Size([2])

In [ ]:

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# ======================================
# 1. 基础设置
# ======================================

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("使用设备：", device)


# ======================================
# 2. 创建词表
# ======================================

vocab = [
    "<PAD>", "<BOS>", "<EOS>",
    "我", "喜欢", "苹果",
    "香蕉", "你", "吃"
]

vocab_size = len(vocab)

token_to_id = {
    token: i for i, token in enumerate(vocab)
}

id_to_token = {
    i: token for i, token in enumerate(vocab)
}


# ======================================
# 3. 训练文本
# ======================================

sentences = [
    ["我", "喜欢", "苹果"],
    ["我", "喜欢", "香蕉"],
    ["你", "吃", "苹果"],
    ["你", "吃", "香蕉"]
]


# ======================================
# 4. 创建 Dataset
# ======================================

class TextDataset(Dataset):

    def __init__(self, sentences):
        self.samples = []

        for sentence in sentences:

            # 添加开始和结束 Token
            tokens = ["<BOS>"] + sentence + ["<EOS>"]

            # Token 转换为 ID
            ids = [
                token_to_id[token]
                for token in tokens
            ]

            # 输入和目标错开一个位置
            x = ids[:-1]
            y = ids[1:]

            self.samples.append((
                torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


dataset = TextDataset(sentences)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)


# ======================================
# 5. 定义 Transformer Block
# ======================================

class TransformerBlock(nn.Module):

    def __init__(self, hidden_size, num_heads):
        super().__init__()

        # Attention 前的 LayerNorm
        self.norm1 = nn.LayerNorm(hidden_size)

        # 多头自注意力
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )

        # FFN 前的 LayerNorm
        self.norm2 = nn.LayerNorm(hidden_size)

        # 前馈神经网络
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2),
            nn.ReLU(),
            nn.Linear(hidden_size * 2, hidden_size)
        )

    def forward(self, x):

        # x: [batch_size, seq_len, hidden_size]

        seq_len = x.shape[1]

        # ----------------------------------
        # 创建因果注意力 Mask
        # ----------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # True 表示禁止关注未来 Token

        # ----------------------------------
        # Pre-LN Self-Attention
        # ----------------------------------

        h = self.norm1(x)

        attn_output, _ = self.attention(
            h, h, h,
            attn_mask=causal_mask,
            need_weights=False
        )

        # 残差连接
        x = x + attn_output

        # ----------------------------------
        # Pre-LN FFN
        # ----------------------------------

        h = self.norm2(x)

        ffn_output = self.ffn(h)

        # 残差连接
        x = x + ffn_output

        return x


# ======================================
# 6. 定义 Mini GPT
# ======================================

class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size=16,
        num_heads=2,
        num_layers=2,
        max_seq_len=16
    ):
        super().__init__()

        self.max_seq_len = max_seq_len

        # Token Embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        # Position Embedding
        self.position_embedding = nn.Embedding(
            max_seq_len,
            hidden_size
        )

        # 多层 Transformer Block
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_size, num_heads)
            for _ in range(num_layers)
        ])

        # 最终 LayerNorm
        self.final_norm = nn.LayerNorm(hidden_size)

        # LM Head
        self.lm_head = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, input_ids):

        # input_ids: [batch_size, seq_len]

        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_seq_len:
            raise ValueError(
                "输入序列长度超过 max_seq_len"
            )

        # ----------------------------------
        # 创建位置索引
        # ----------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        # ----------------------------------
        # Embedding
        # ----------------------------------

        token_emb = self.token_embedding(input_ids)

        pos_emb = self.position_embedding(positions)

        # [batch_size, seq_len, hidden_size]
        x = token_emb + pos_emb

        # ----------------------------------
        # 多层 Transformer
        # ----------------------------------

        for block in self.blocks:
            x = block(x)

        # 最终归一化
        x = self.final_norm(x)

        # ----------------------------------
        # LM Head
        # ----------------------------------

        logits = self.lm_head(x)

        # [batch_size, seq_len, vocab_size]
        return logits


# ======================================
# 7. 创建模型、损失函数、优化器
# ======================================

model = MiniGPT(
    vocab_size=vocab_size,
    hidden_size=16,
    num_heads=2,
    num_layers=2,
    max_seq_len=16
).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

print("模型参数量：", sum(
    p.numel() for p in model.parameters()
))


# ======================================
# 8. 模型训练
# ======================================

epochs = 10000

for epoch in range(epochs):

    model.train()

    total_loss = 0.0
    total_tokens = 0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        # 清空梯度
        optimizer.zero_grad()

        # 前向传播
        logits = model(x)

        # logits: [batch_size, seq_len, vocab_size]

        # 计算交叉熵损失
        loss = loss_fn(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        # 反向传播
        loss.backward()

        # 更新参数
        optimizer.step()

        # 统计当前 Batch 的 Token 数量
        token_count = y.numel()

        # 累加损失
        total_loss += loss.item() * token_count
        total_tokens += token_count

    avg_loss = total_loss / total_tokens

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss: {avg_loss:.4f}"
        )


# ======================================
# 9. 文本生成
# ======================================

@torch.no_grad()
def generate(model, prompt, max_new_tokens=10):

    model.eval()

    # 添加 BOS
    tokens = ["<BOS>"] + prompt

    # 转换为 Token ID
    ids = [
        token_to_id[token]
        for token in tokens
    ]

    for _ in range(max_new_tokens):

        # ----------------------------------
        # 准备输入
        # ----------------------------------

        # 超过最大长度时，保留最近的 Token
        context_ids = ids[-model.max_seq_len:]

        x = torch.tensor(
            [context_ids],
            dtype=torch.long,
            device=device
        )

        # ----------------------------------
        # 前向传播
        # ----------------------------------

        logits = model(x)

        # 取最后一个位置的 Logits
        last_logits = logits[:, -1, :]

        # ----------------------------------
        # 贪心选择
        # ----------------------------------

        next_id = torch.argmax(
            last_logits,
            dim=-1
        ).item()

        # 生成 EOS 时停止
        if next_id == token_to_id["<EOS>"]:
            break

        # 添加生成的 Token
        ids.append(next_id)

    # Token ID 转换为文本
    return [
        id_to_token[i]
        for i in ids
        if id_to_token[i] not in (
            "<BOS>", "<EOS>", "<PAD>"
        )
    ]


# ======================================
# 10. 测试文本生成
# ======================================

print("\n生成结果：")

print(
    "输入：我 喜欢",
    "输出：",
    generate(model, ["我", "喜欢"])
)

print(
    "输入：你 吃",
    "输出：",
    generate(model, ["你", "吃"])
)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model.eval()

print("设备:", device)
print("模型类型:", type(model).__name__)
print("词表大小:", model.config.vocab_size)
print("隐藏维度:", model.config.n_embd)
print("Transformer 层数:", model.config.n_layer)
print("Attention Head 数量:", model.config.n_head)

In [ ]:

text = "I love apples"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs)
print("input_ids:", inputs["input_ids"])
print("attention_mask:", inputs["attention_mask"])
print("形状:", inputs["input_ids"].shape)

# 把 Token ID 转换回 Token 字符串
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print("Tokens:", tokens)


In [1]:

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ==========================================
# 1. 加载 Tokenizer 和预训练模型
# ==========================================

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 默认没有独立的 PAD Token
# 教学中复用 EOS Token 作为 PAD Token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
).to(device)

# ==========================================
# 2. 准备训练文本
# ==========================================

texts = [
    "The cat is sleeping.",
    "The dog is running.",
    "The cat likes milk.",
    "The dog likes food."
]

# ==========================================
# 3. 创建 Dataset
# ==========================================

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]


dataset = TextDataset(texts)

# ==========================================
# 4. 定义 Collate Function
# ==========================================

def collate_fn(batch_texts):
    # 每个句子添加结束标记
    batch_texts = [
        text + tokenizer.eos_token
        for text in batch_texts
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # labels 和 input_ids 初始相同
    labels = input_ids.clone()

    # 忽略 Padding 位置
    labels[attention_mask == 0] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# ==========================================
# 5. 创建优化器
# ==========================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

# ==========================================
# 6. 训练
# ==========================================

epochs = 300

for epoch in range(epochs):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in dataloader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        # Hugging Face 自动计算 Loss
        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        # 统计有效预测目标的数量
        valid_targets = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        total_loss += loss.item() * valid_targets
        total_tokens += valid_targets

    avg_loss = total_loss / total_tokens

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {avg_loss:.4f}"
    )

# ==========================================
# 7. 保存微调模型
# ==========================================

save_dir = "./my_finetuned_gpt2"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("模型保存完成")

# ==========================================
# 8. 重新加载模型
# ==========================================

loaded_tokenizer = AutoTokenizer.from_pretrained(
    save_dir
)

loaded_model = AutoModelForCausalLM.from_pretrained(
    save_dir
).to(device)

loaded_model.eval()

# ==========================================
# 9. 生成文本
# ==========================================

prompt = "The cat"

inputs = loaded_tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    generated_ids = loaded_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=loaded_tokenizer.eos_token_id
    )

generated_text = loaded_tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("生成结果:")
print(generated_text)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 | Loss: 6.4894
Epoch 2 | Loss: 5.7205
Epoch 3 | Loss: 5.1770
Epoch 4 | Loss: 4.3882
Epoch 5 | Loss: 4.1317
Epoch 6 | Loss: 3.9230
Epoch 7 | Loss: 3.3461
Epoch 8 | Loss: 3.2899
Epoch 9 | Loss: 3.1123
Epoch 10 | Loss: 2.8100
Epoch 11 | Loss: 2.6201
Epoch 12 | Loss: 2.5956
Epoch 13 | Loss: 2.0832
Epoch 14 | Loss: 2.2939
Epoch 15 | Loss: 2.0234
Epoch 16 | Loss: 2.1650
Epoch 17 | Loss: 1.8739
Epoch 18 | Loss: 1.8864
Epoch 19 | Loss: 2.1204
Epoch 20 | Loss: 2.0965
Epoch 21 | Loss: 1.8079
Epoch 22 | Loss: 1.7368
Epoch 23 | Loss: 1.6007
Epoch 24 | Loss: 1.6686
Epoch 25 | Loss: 2.0441
Epoch 26 | Loss: 1.4215
Epoch 27 | Loss: 1.6491
Epoch 28 | Loss: 1.2779
Epoch 29 | Loss: 1.4167
Epoch 30 | Loss: 1.3363
Epoch 31 | Loss: 1.0735
Epoch 32 | Loss: 1.4697
Epoch 33 | Loss: 1.4335
Epoch 34 | Loss: 0.9687
Epoch 35 | Loss: 1.2956
Epoch 36 | Loss: 1.2091
Epoch 37 | Loss: 0.9206
Epoch 38 | Loss: 1.1500
Epoch 39 | Loss: 0.8714
Epoch 40 | Loss: 1.1572
Epoch 41 | Loss: 0.9806
Epoch 42 | Loss: 0.7872
E

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型保存完成


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

生成结果:
The cat likes milk.


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

tokenizer.pad_token = tokenizer.eos_token

texts = [
    "The cat is sleeping.",
    "The dog runs.",
    "I like apples.",
    "Hello."
]

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]


def collate_fn(batch_texts):
    # 给每句话添加 EOS
    batch_texts = [
        text + tokenizer.eos_token
        for text in batch_texts
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        return_tensors="pt"
    )

    labels = encoded["input_ids"].clone()

    # 只忽略真实 Padding 位置
    labels[encoded["attention_mask"] == 0] = -100

    encoded["labels"] = labels

    return encoded


dataset = TextDataset(texts)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

for batch_index, batch in enumerate(dataloader):
    print(f"\n========== Batch {batch_index} ==========")

    print("input_ids:")
    print(batch["input_ids"])

    print("attention_mask:")
    print(batch["attention_mask"])

    print("labels:")
    print(batch["labels"])

    print("input_ids shape:", batch["input_ids"].shape)
    print("labels shape:", batch["labels"].shape)

    print("有效预测目标数量:")
    print((batch["labels"][:, 1:] != -100).sum().item())

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ==========================================
# 1. 加载 Tokenizer 和预训练模型
# ==========================================

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 默认没有独立的 PAD Token
# 教学中复用 EOS Token 作为 PAD Token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
).to(device)

# ==========================================
# 2. 准备训练文本
# ==========================================

train_texts = [
    "The cat is sleeping.",
    "The dog is running.",
    "The cat likes milk.",
    "The dog likes food.",
    "I like apples.",
    "I like bananas."
]

eval_texts = [
    "The cat is running.",
    "The dog likes milk."
]

# ==========================================
# 3. 创建 Dataset
# ==========================================

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]


train_dataset = TextDataset(train_texts)
eval_dataset = TextDataset(eval_texts)

# ==========================================
# 4. 定义 Collate Function
# ==========================================

def collate_fn(batch_texts):
    # 每个句子添加结束标记
    batch_texts = [
        text + tokenizer.eos_token
        for text in batch_texts
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    labels = encoded["input_ids"].clone()

    # 忽略 Padding 位置
    labels[encoded["attention_mask"] == 0] = -100

    encoded["labels"] = labels

    return encoded


args = TrainingArguments(
    output_dir="./trainer_results", # 保存模型检查点等训练输出的目录

    per_device_train_batch_size=2, # 每张 GPU 每次前向/反向传播处理 2 个训练样本
    per_device_eval_batch_size=2, # 每张 GPU 每次验证处理 2 个样本

    gradient_accumulation_steps=2, # 累积 2 次梯度后更新一次模型参数

    num_train_epochs=300, # 完整遍历训练集 3 次
    learning_rate=2e-5, # 初始学习率，即 0.00002

    eval_strategy="epoch", # 每个 Epoch 结束后验证一次
    save_strategy="epoch", # 每个 Epoch 结束后保存一次检查点

    logging_strategy="steps", # 按训练步数记录日志
    logging_steps=1, # 每 1 个训练更新步记录一次日志

    save_total_limit=2, # 限制保留的检查点数量，通常最多保留 2 个

    report_to="none", # 不向 TensorBoard、W&B 等外部平台报告

    remove_unused_columns=False # 不自动删除 Dataset 中模型 forward() 未使用的字段
)

# 创建 Trainer
trainer = Trainer(
    model=model,
    args=args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=collate_fn
)

# 开始训练
trainer.train()

# 验证
metrics = trainer.evaluate()

print("验证结果:")
print(metrics)

# ======================================
# 9. 保存模型
# ======================================

save_dir = "./my_trainer_gpt2"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("模型保存完成")

# ======================================
# 10. 加载并生成文本
# ======================================

loaded_model = AutoModelForCausalLM.from_pretrained(
    save_dir
)

loaded_tokenizer = AutoTokenizer.from_pretrained(
    save_dir
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

inputs = loaded_tokenizer(
    "The cat",
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = loaded_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=loaded_tokenizer.eos_token_id
    )

print(
    loaded_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
)


In [48]:

import torch
from torch import nn

torch.manual_seed(42)

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=2, alpha=4):
        super().__init__()

        # 原始线性层
        self.base = nn.Linear(
            in_features,
            out_features,
            bias=False
        )

        # 冻结原始权重
        for param in self.base.parameters():
            param.requires_grad = False

        # LoRA A: in_features -> r
        self.lora_A = nn.Linear(
            in_features,
            r,
            bias=False
        )

        # LoRA B: r -> out_features
        self.lora_B = nn.Linear(
            r,
            out_features,
            bias=False
        )

        self.scale = alpha / r

        # 初始化：A 随机，B 为零
        nn.init.normal_(
            self.lora_A.weight,
            std=0.02
        )

        nn.init.zeros_(
            self.lora_B.weight
        )

        print("LoRA A: ", self.lora_A.weight)
        print("LoRA B: ", self.lora_B.weight)

    def forward(self, x):
        # 原模型输出
        base_output = self.base(x)

        # LoRA 分支
        lora_output = self.lora_B(
            self.lora_A(x)
        ) * self.scale

        return base_output + lora_output


model = LoRALinear(
    in_features=3,
    out_features=2,
    r=2,
    alpha=4
)

x = torch.tensor([[1.0, 2.0, 3.0]])

output = model(x)

print("输出:", output)

print("\n参数状态:")
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)

print("\n可训练参数数量:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))


LoRA A:  Parameter containing:
tensor([[ 0.0092,  0.0053,  0.0107],
        [ 0.0162,  0.0222, -0.0338]], requires_grad=True)
LoRA B:  Parameter containing:
tensor([[0., 0.],
        [0., 0.]], requires_grad=True)
输出: tensor([[0.9940, 0.6269]], grad_fn=<AddBackward0>)

参数状态:
base.weight torch.Size([2, 3]) False
lora_A.weight torch.Size([2, 3]) True
lora_B.weight torch.Size([2, 2]) True

可训练参数数量:
10


In [57]:

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ==========================================
# 1. 加载预训练模型
# ==========================================

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# ==========================================
# 2. 配置 LoRA
# ==========================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    target_modules=["c_attn"],

    bias="none",

    task_type=TaskType.CAUSAL_LM,

    fan_in_fan_out=True
)

# ==========================================
# 3. 将 LoRA 添加到模型
# ==========================================

model = get_peft_model(
    base_model,
    lora_config
)

model = model.to(device)

print("LoRA 参数统计:")
model.print_trainable_parameters()

# ==========================================
# 4. 训练数据
# ==========================================

texts = [
    "The cat is sleeping.",
    "The dog is running.",
    "The cat likes milk.",
    "The dog likes food.",
    "I like apples.",
    "I like bananas."
]

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]


dataset = TextDataset(texts)

# ==========================================
# 5. Collate Function
# ==========================================

def collate_fn(batch_texts):
    batch_texts = [
        text + tokenizer.eos_token
        for text in batch_texts
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    labels = encoded["input_ids"].clone()

    labels[encoded["attention_mask"] == 0] = -100

    encoded["labels"] = labels

    return encoded


dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# ==========================================
# 6. 优化器：只传入可训练参数
# ==========================================

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-4
)

# ==========================================
# 7. LoRA 微调
# ==========================================

epochs = 200

for epoch in range(epochs):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in dataloader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        valid_targets = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        total_loss += loss.item() * valid_targets
        total_tokens += valid_targets

    avg_loss = total_loss / total_tokens

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {avg_loss:.4f}"
    )

# ==========================================
# 8. 保存 LoRA Adapter
# ==========================================

adapter_dir = "./my_lora_adapter"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("LoRA Adapter 保存完成")

# ==========================================
# 9. 生成文本
# ==========================================

model.eval()

inputs = tokenizer(
    "The cat",
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

LoRA 参数统计:
trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797
Epoch 1 | Loss: 6.6293
Epoch 2 | Loss: 6.6056
Epoch 3 | Loss: 6.5402
Epoch 4 | Loss: 6.4752
Epoch 5 | Loss: 6.4577
Epoch 6 | Loss: 6.1952
Epoch 7 | Loss: 6.3283
Epoch 8 | Loss: 6.4292
Epoch 9 | Loss: 6.3650
Epoch 10 | Loss: 6.1613
Epoch 11 | Loss: 5.9753
Epoch 12 | Loss: 6.1304
Epoch 13 | Loss: 6.2050
Epoch 14 | Loss: 5.9653
Epoch 15 | Loss: 5.7630
Epoch 16 | Loss: 5.7353
Epoch 17 | Loss: 5.5785
Epoch 18 | Loss: 5.6839
Epoch 19 | Loss: 5.5620
Epoch 20 | Loss: 5.4414
Epoch 21 | Loss: 5.2687
Epoch 22 | Loss: 5.1859
Epoch 23 | Loss: 5.0809
Epoch 24 | Loss: 4.9778
Epoch 25 | Loss: 4.8197
Epoch 26 | Loss: 4.8601
Epoch 27 | Loss: 4.5718
Epoch 28 | Loss: 4.5525
Epoch 29 | Loss: 4.5940
Epoch 30 | Loss: 4.3190
Epoch 31 | Loss: 4.2726
Epoch 32 | Loss: 4.1534
Epoch 33 | Loss: 4.1071
Epoch 34 | Loss: 4.1093
Epoch 35 | Loss: 4.1019
Epoch 36 | Loss: 3.8746
Epoch 37 | Loss: 3.8643
Epoch 38 | Loss: 3.7522
Epoch 39 | L

In [ ]:
# QLoRA
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType
)

torch.manual_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("本示例需要支持 4-bit 量化的 CUDA GPU")

device = torch.device("cuda")

# =====================================
# 1. 配置 4-bit 量化
# =====================================

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

# =====================================
# 2. 加载 Tokenizer
# =====================================

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# =====================================
# 3. 加载量化基础模型
# =====================================

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)

# =====================================
# 4. 为低比特训练做准备
# =====================================

base_model = prepare_model_for_kbit_training(
    base_model
)

# =====================================
# 5. 添加 LoRA
# =====================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    fan_in_fan_out=True
)

model = get_peft_model(
    base_model,
    lora_config
)

model.print_trainable_parameters()

# =====================================
# 6. 数据集
# =====================================

texts = [
    "The cat is sleeping.",
    "The dog is running.",
    "The cat likes milk.",
    "The dog likes food.",
    "I like apples.",
    "I like bananas."
]

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]

dataset = TextDataset(texts)

# =====================================
# 7. 整理 Batch
# =====================================

def collate_fn(batch_texts):
    batch_texts = [
        text + tokenizer.eos_token
        for text in batch_texts
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    labels = encoded["input_ids"].clone()

    # 忽略 Padding，保留真实 EOS
    labels[encoded["attention_mask"] == 0] = -100

    encoded["labels"] = labels

    return encoded

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# =====================================
# 8. 优化器
# =====================================

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

# =====================================
# 9. 训练
# =====================================

epochs = 3

for epoch in range(epochs):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in dataloader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        valid_targets = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        total_loss += loss.item() * valid_targets
        total_tokens += valid_targets

    avg_loss = total_loss / total_tokens

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {avg_loss:.4f}"
    )

# =====================================
# 10. 保存 Adapter
# =====================================

adapter_dir = "./my_qlora_adapter"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("QLoRA Adapter 保存完成")

# =====================================
# 11. 重新加载量化模型与 Adapter
# =====================================

del model
del base_model
torch.cuda.empty_cache()

reloaded_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)

loaded_model = PeftModel.from_pretrained(
    reloaded_base,
    adapter_dir
)

loaded_model.eval()

# =====================================
# 12. 推理
# =====================================

inputs = tokenizer(
    "The cat",
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = loaded_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
)


In [9]:

import random
import numpy as np
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# ======================================
# 1. 固定随机种子
# ======================================

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ======================================
# 2. 准备教学数据
# ======================================

positive_texts = [
    "This movie is great.",
    "This film is wonderful.",
    "I love this movie.",
    "The acting is amazing.",
    "The story is excellent.",
    "This is a fantastic film.",
    "I really enjoyed the movie.",
    "The ending was beautiful.",
    "The characters are lovable.",
    "The film is very entertaining.",
    "What a brilliant performance.",
    "The movie made me happy.",
    "I would watch it again.",
    "The plot is interesting.",
    "Everything about this film is good.",
    "I highly recommend this movie.",
    "The soundtrack is beautiful.",
    "The movie is impressive.",
    "This is an enjoyable experience.",
    "I am very satisfied with the film."
]

negative_texts = [
    "This movie is terrible.",
    "This film is boring.",
    "I hate this movie.",
    "The acting is awful.",
    "The story is disappointing.",
    "This is a horrible film.",
    "I really disliked the movie.",
    "The ending was terrible.",
    "The characters are annoying.",
    "The film is very dull.",
    "What a poor performance.",
    "The movie made me angry.",
    "I would never watch it again.",
    "The plot is confusing.",
    "Everything about this film is bad.",
    "I do not recommend this movie.",
    "The soundtrack is annoying.",
    "The movie is disappointing.",
    "This is an unpleasant experience.",
    "I am very dissatisfied with the film."
]

texts = positive_texts + negative_texts

labels = (
    [1] * len(positive_texts)
    + [0] * len(negative_texts)
)

# ======================================
# 3. Train / Validation / Test
# ======================================

train_texts, temp_texts, train_labels, temp_labels = (
    train_test_split(
        texts,
        labels,
        test_size=0.4,
        random_state=seed,
        stratify=labels
    )
)

val_texts, test_texts, val_labels, test_labels = (
    train_test_split(
        temp_texts,
        temp_labels,
        test_size=0.5,
        random_state=seed,
        stratify=temp_labels
    )
)

print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

# ======================================
# 4. 加载 Tokenizer
# ======================================

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# ======================================
# 5. 创建 Dataset
# ======================================

class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=64
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return item


train_dataset = ClassificationDataset(
    train_texts,
    train_labels,
    tokenizer
)

val_dataset = ClassificationDataset(
    val_texts,
    val_labels,
    tokenizer
)

test_dataset = ClassificationDataset(
    test_texts,
    test_labels,
    tokenizer
)

# ======================================
# 6. 加载分类模型
# ======================================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={
        0: "NEGATIVE",
        1: "POSITIVE"
    },
    label2id={
        "NEGATIVE": 0,
        "POSITIVE": 1
    }
)

# ======================================
# 7. 评估指标
# ======================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ======================================
# 8. TrainingArguments
# ======================================

args = TrainingArguments(
    output_dir="./classification_results",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=1,

    num_train_epochs=10,
    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="epoch",

    save_total_limit=2,

    report_to="none",

    seed=seed
)

# ======================================
# 9. 创建 Trainer
# ======================================

trainer = Trainer(
    model=model,
    args=args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

# ======================================
# 10. 训练
# ======================================

trainer.train()

# ======================================
# 11.
# ======================================

print(
    "Best Checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best Metric:",
    trainer.state.best_metric
)

# ======================================
# 12. 验证集评估
# ======================================

val_metrics = trainer.evaluate(
    eval_dataset=val_dataset
)

print("Validation Metrics:")
print(val_metrics)

# ======================================
# 13. 测试集最终评估
# ======================================

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

print("Test Metrics:")
print(test_metrics)

# ======================================
# 14. 混淆矩阵
# ======================================

test_output = trainer.predict(
    test_dataset
)

test_predictions = np.argmax(
    test_output.predictions,
    axis=-1
)

cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm)

# ======================================
# 15. 保存最佳模型
# ======================================

save_dir = "./best_classifier"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Best model saved:", save_dir)


Train: 24
Validation: 8
Test: 8


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.694561,0.709491,0.375000,0.428571,0.750000,0.545455
2,0.651989,0.707346,0.375000,0.428571,0.750000,0.545455
3,0.627981,0.707050,0.625000,0.600000,0.750000,0.666667
4,0.559925,0.692271,0.625000,0.600000,0.750000,0.666667
5,0.483466,0.669513,0.625000,0.600000,0.750000,0.666667
6,0.416033,0.630123,0.625000,0.600000,0.750000,0.666667
7,0.333678,0.597475,0.625000,0.600000,0.750000,0.666667
8,0.296532,0.576835,0.625000,0.600000,0.750000,0.666667
9,0.256747,0.563504,0.625000,0.600000,0.750000,0.666667
10,0.241745,0.558195,0.625000,0.600000,0.750000,0.666667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (13 outputs left out)

Test Metrics:
{'test_loss': 0.42808419466018677, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1': 1.0}


Confusion Matrix:
[[4 0]
 [0 4]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved: ./best_classifier


In [10]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

save_dir = "./best_classifier"

tokenizer = AutoTokenizer.from_pretrained(
    save_dir
)

model = AutoModelForSequenceClassification.from_pretrained(
    save_dir
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

text = "This movie is absolutely wonderful."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=64
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

probabilities = torch.softmax(
    logits,
    dim=-1
)

predicted_class = logits.argmax(
    dim=-1
).item()

print("Text:", text)

print(
    "Prediction:",
    model.config.id2label[predicted_class]
)

print(
    "Probabilities:",
    probabilities.cpu().tolist()
)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This movie is absolutely wonderful.
Prediction: POSITIVE
Probabilities: [[0.2051011323928833, 0.7948988676071167]]


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
).to(device)

# =====================================
# 1. SFT 数据
# =====================================

samples = [
    {
        "prompt": "Question: What is 2 + 3?\nAnswer:",
        "completion": " 5"
    },
    {
        "prompt": "Question: What is 4 + 6?\nAnswer:",
        "completion": " 10"
    },
    {
        "prompt": "Question: What is 1 + 1?\nAnswer:",
        "completion": " 2"
    },
    {
        "prompt": "Question: What is 3 + 4?\nAnswer:",
        "completion": " 7"
    }
]

# =====================================
# 2. Dataset
# =====================================

class SFTDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.items = []

        for sample in samples:
            prompt = sample["prompt"]
            completion = sample["completion"]

            prompt_ids = tokenizer(
                prompt,
                add_special_tokens=False
            )["input_ids"]

            # 分别编码，再拼接，保证边界明确
            completion_ids = tokenizer(
                completion,
                add_special_tokens=False
            )["input_ids"]

            completion_ids += [
                tokenizer.eos_token_id
            ]

            input_ids = (
                prompt_ids + completion_ids
            )

            labels = (
                [-100] * len(prompt_ids)
                + completion_ids
            )

            self.items.append({
                "input_ids": input_ids,
                "labels": labels
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

dataset = SFTDataset(
    samples,
    tokenizer
)

# =====================================
# 3. Collate Function
# =====================================

def collate_fn(batch):
    max_len = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids_list = []
    attention_masks = []
    labels_list = []

    for item in batch:
        ids = item["input_ids"]
        labels = item["labels"]

        pad_len = max_len - len(ids)

        padded_ids = (
            ids
            + [tokenizer.pad_token_id] * pad_len
        )

        padded_labels = (
            labels
            + [-100] * pad_len
        )

        mask = (
            [1] * len(ids)
            + [0] * pad_len
        )

        input_ids_list.append(padded_ids)
        attention_masks.append(mask)
        labels_list.append(padded_labels)

    return {
        "input_ids": torch.tensor(
            input_ids_list,
            dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            attention_masks,
            dtype=torch.long
        ),
        "labels": torch.tensor(
            labels_list,
            dtype=torch.long
        )
    }

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# =====================================
# 4. 查看一条样本
# =====================================

item = dataset[0]

print("Input IDs:")
print(item["input_ids"])

print("Labels:")
print(item["labels"])

print("完整文本:")
print(tokenizer.decode(item["input_ids"]))

# =====================================
# 5. 优化器
# =====================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5
)

# =====================================
# 6. 训练
# =====================================

epochs = 3

for epoch in range(epochs):
    model.train()

    total_loss = 0.0
    total_steps = 0

    for batch in dataloader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_steps += 1

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {total_loss / total_steps:.4f}"
    )

# =====================================
# 7. 推理
# =====================================

model.eval()

prompt = "Question: What is 2 + 3?\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
)


In [ ]:

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from trl import (
    SFTTrainer,
    SFTConfig
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

dataset = Dataset.from_list([
    {
        "prompt": "Question: What is 2 + 3?\nAnswer:",
        "completion": " 5"
    },
    {
        "prompt": "Question: What is 4 + 6?\nAnswer:",
        "completion": " 10"
    },
    {
        "prompt": "Question: What is 1 + 1?\nAnswer:",
        "completion": " 2"
    },
    {
        "prompt": "Question: What is 3 + 4?\nAnswer:",
        "completion": " 7"
    }
])

args = SFTConfig(
    output_dir="./sft_results",

    per_device_train_batch_size=2,

    num_train_epochs=3,

    learning_rate=5e-5,

    max_length=128,

    completion_only_loss=True,

    packing=False,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

trainer.save_model("./my_sft_model")
tokenizer.save_pretrained("./my_sft_model")


In [11]:

from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

messages = [
    {
        "role": "system",
        "content": "你是一个数学助手。"
    },
    {
        "role": "user",
        "content": "2 + 3 等于多少？"
    },
    {
        "role": "assistant",
        "content": "5。"
    },
    {
        "role": "user",
        "content": "再加 4 呢？"
    },
    {
        "role": "assistant",
        "content": "9。"
    }
]

# 1. 格式化字符串
formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("========== 格式化文本 ==========")
print(formatted_text)

# 2. 转换成 Token ID
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print("\n========== Token ID ==========")
print(input_ids)

# 3. 查看每个 Token
print("\n========== Token 明细 ==========")

for index, token_id in enumerate(input_ids):
    token = tokenizer.convert_ids_to_tokens(
        token_id
    )

    print(
        f"位置 {index:02d} | "
        f"ID {token_id:>8} | "
        f"Token {repr(token)}"
    )

# 4. 还原文本
decoded = tokenizer.decode(
    input_ids,
    skip_special_tokens=False
)

print("\n========== Decode ==========")
print(decoded)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

========== 格式化文本 ==========
<|im_start|>system
你是一个数学助手。<|im_end|>
<|im_start|>user
2 + 3 等于多少？<|im_end|>
<|im_start|>assistant
5。<|im_end|>
<|im_start|>user
再加 4 呢？<|im_end|>
<|im_start|>assistant
9。<|im_end|>


========== Token ID ==========
{'input_ids': [151644, 8948, 198, 56568, 101909, 104552, 110498, 1773, 151645, 198, 151644, 872, 198, 17, 488, 220, 18, 10236, 255, 231, 34204, 100430, 11319, 151645, 198, 151644, 77091, 198, 20, 1773, 151645, 198, 151644, 872, 198, 87256, 20929, 220, 19, 4891, 239, 95, 11319, 151645, 198, 151644, 77091, 198, 24, 1773, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

========== Token 明细 ==========


ValueError: invalid literal for int() with base 10: 'i'

In [ ]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

messages = [
    {
        "role": "system",
        "content": "你是一个数学助手。"
    },
    {
        "role": "user",
        "content": "2 + 3 等于多少？"
    },
    {
        "role": "assistant",
        "content": "5。"
    },
    {
        "role": "user",
        "content": "再加 4 呢？"
    },
    {
        "role": "assistant",
        "content": "9。"
    }
]

input_ids = []
labels = []

# 每次只格式化一个消息，并与前缀比较
# 这样可以观察消息边界是否符合预期
for i, message in enumerate(messages):

    prefix_ids = tokenizer.apply_chat_template(
        messages[:i],
        tokenize=True,
        add_generation_prompt=False
    ) if i > 0 else []

    current_ids = tokenizer.apply_chat_template(
        messages[:i + 1],
        tokenize=True,
        add_generation_prompt=False
    )

    # 要求前面的完整消息不被模板重写
    if current_ids[:len(prefix_ids)] != prefix_ids:
        raise ValueError(
            "当前 Chat Template 不满足前缀稳定性，"
            "不能使用这种简单的差分方法。"
        )

    new_ids = current_ids[len(prefix_ids):]

    input_ids.extend(new_ids)

    if message["role"] == "assistant":
        # 教学简化：整个 Assistant 消息新增部分参与 Loss
        labels.extend(new_ids)
    else:
        labels.extend(
            [-100] * len(new_ids)
        )

input_ids = torch.tensor(
    [input_ids],
    dtype=torch.long
)

labels = torch.tensor(
    [labels],
    dtype=torch.long
)

attention_mask = torch.ones_like(
    input_ids
)

print("Input shape:", input_ids.shape)
print("Labels shape:", labels.shape)

print("\n========== 监督位置 ==========")

for token_id, label in zip(
    input_ids[0].tolist(),
    labels[0].tolist()
):
    token = tokenizer.convert_ids_to_tokens(
        token_id
    )

    print(
        f"{repr(token):25} "
        f"label={label}"
    )

# 只做一次前向传播，观察 Loss
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("\nLoss:", outputs.loss.item())


In [ ]:

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from trl import (
    SFTTrainer,
    SFTConfig
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

dataset = Dataset.from_list([
    {
        "messages": [
            {
                "role": "system",
                "content": "你是一个数学助手。"
            },
            {
                "role": "user",
                "content": "2 + 3 等于多少？"
            },
            {
                "role": "assistant",
                "content": "5。"
            },
            {
                "role": "user",
                "content": "再加 4 呢？"
            },
            {
                "role": "assistant",
                "content": "9。"
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "10 - 4 等于多少？"
            },
            {
                "role": "assistant",
                "content": "6。"
            }
        ]
    }
])

args = SFTConfig(
    output_dir="./multi_turn_sft",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=2,

    num_train_epochs=3,

    learning_rate=2e-5,

    max_length=256,

    # 不依赖模板生成区域标记
    assistant_only_loss=False,

    packing=False,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

trainer.save_model("./multi_turn_model")
tokenizer.save_pretrained("./multi_turn_model")


In [5]:
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# SFT 数据
samples = [
    {
        "prompt": "Question: What is 2 + 3?\nAnswer:",
        "completion": " 5"
    },
    {
        "prompt": "Question: What is 4 + 6?\nAnswer:",
        "completion": " 10"
    },
    {
        "prompt": "Question: What is 1 + 1?\nAnswer:",
        "completion": " 2"
    },
    {
        "prompt": "Question: What is 3 + 4?\nAnswer:",
        "completion": " 7"
    }
]

# Dataset
class SFTDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.items = []

        for sample in samples:
            prompt = sample['prompt']
            completion = sample['completion']

            prompt_ids = tokenizer(
                prompt,
                add_special_tokens=False,
            )['input_ids']

            completion_ids = tokenizer(
                completion,
                add_special_tokens=False,
            )['input_ids']

            completion_ids += [tokenizer.eos_token_id]

            input_ids = prompt_ids + completion_ids

            labels = [-100] * len(prompt_ids) + completion_ids

            self.items.append({
                'input_ids': input_ids,
                'labels': labels
            })

    def __len__(self):
        return len(self.items)


    def __getitem__(self, index):
        return self.items[index]

dataset = SFTDataset(
    samples=samples,
    tokenizer=tokenizer
)

# Collate Function
def collate_fn(batch):
    max_len = max(len(item['input_ids']) for item in batch)

    input_ids_list = []
    attention_masks = []
    labels_list = []

    for item in batch:
        ids = item['input_ids']
        labels = item['labels']

        pad_len = max_len - len(ids)

        padded_ids = ids + [-100] * pad_len
        padded_labels = labels + [-100] * pad_len

        mask = [1] * len(ids) + [0] * pad_len

        input_ids_list.append(padded_ids)
        attention_masks.append(mask)
        labels_list.append(padded_labels)

    return {
        "input_ids": torch.tensor(
            input_ids_list,
            dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            attention_masks,
            dtype=torch.long
        ),
        "labels": torch.tensor(
            labels_list,
            dtype=torch.long
        )
    }

dataloader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=collate_fn
    )

item = dataset[0]

print("Input IDs:")
print(item["input_ids"])

print("Labels:")
print(item["labels"])

print("完整文本:")
print(tokenizer.decode(item["input_ids"]))


# 优化器
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
)

# 训练
epochs = 100

for epoch in range(epochs):
    model.train()

    total_loss = 0.0
    total_steps = 0

    for batch in dataloader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        total_steps += 1

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {total_loss / total_steps:.4f}"
    )

# 推理
model.eval()

prompt = "Question: What is 12 + 13?\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Input IDs:
[24361, 25, 1867, 318, 362, 1343, 513, 30, 198, 33706, 25, 642, 50256]
Labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 642, 50256]
完整文本:
Question: What is 2 + 3?
Answer: 5<|endoftext|>
Epoch 1 | Loss: 5.1126
Epoch 2 | Loss: 3.5372
Epoch 3 | Loss: 2.4341
Epoch 4 | Loss: 1.8108
Epoch 5 | Loss: 1.4927
Epoch 6 | Loss: 1.2823
Epoch 7 | Loss: 1.1816
Epoch 8 | Loss: 0.9339
Epoch 9 | Loss: 0.8424
Epoch 10 | Loss: 0.7642
Epoch 11 | Loss: 0.6377
Epoch 12 | Loss: 0.6382
Epoch 13 | Loss: 0.5257
Epoch 14 | Loss: 0.5093
Epoch 15 | Loss: 0.4893
Epoch 16 | Loss: 0.4404
Epoch 17 | Loss: 0.5007
Epoch 18 | Loss: 0.4680
Epoch 19 | Loss: 0.4744
Epoch 20 | Loss: 0.3724
Epoch 21 | Loss: 0.2995
Epoch 22 | Loss: 0.4488
Epoch 23 | Loss: 0.2494
Epoch 24 | Loss: 0.2275
Epoch 25 | Loss: 0.1872
Epoch 26 | Loss: 1.0357
Epoch 27 | Loss: 0.2454
Epoch 28 | Loss: 0.3351
Epoch 29 | Loss: 0.3802
Epoch 30 | Loss: 0.0531
Epoch 31 | Loss: 0.1058
Epoch 32 | Loss: 0.3622
Epoch 33 | Loss: 0

In [6]:

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from trl import (
    SFTTrainer,
    SFTConfig
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

dataset = Dataset.from_list([
    {
        "prompt": "Question: What is 2 + 3?\nAnswer:",
        "completion": " 5"
    },
    {
        "prompt": "Question: What is 4 + 6?\nAnswer:",
        "completion": " 10"
    },
    {
        "prompt": "Question: What is 1 + 1?\nAnswer:",
        "completion": " 2"
    },
    {
        "prompt": "Question: What is 3 + 4?\nAnswer:",
        "completion": " 7"
    }
])

args = SFTConfig(
    output_dir="./sft_results",

    per_device_train_batch_size=2,

    num_train_epochs=3,

    learning_rate=5e-5,

    max_length=128,

    completion_only_loss=True,

    packing=False,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

trainer.save_model("./my_sft_model")
tokenizer.save_pretrained("./my_sft_model")


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

In file included from /home/niyuhang/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu/include/python3.12/Python.h:12,
                 from /home/niyuhang/workspace/py_workspace/02_huggingface/.venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:9:
/home/niyuhang/.local/share/uv/python/cpython-3.12.16-linux-x86_64-gnu/include/python3.12/pyconfig.h:1877:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1877 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/niyuhang/workspace/py_workspace/02_huggingface/.venv/lib/python3.12/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/niyuhang/workspace/py_workspace/02_huggingface/.venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_sft_model/tokenizer_config.json', './my_sft_model/tokenizer.json')

In [7]:

from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

messages = [
    {
        "role": "system",
        "content": "你是一个数学助手。"
    },
    {
        "role": "user",
        "content": "2 + 3 等于多少？"
    },
    {
        "role": "assistant",
        "content": "5。"
    },
    {
        "role": "user",
        "content": "再加 4 呢？"
    },
    {
        "role": "assistant",
        "content": "9。"
    }
]

# 1. 格式化字符串
formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("========== 格式化文本 ==========")
print(formatted_text)

# 2. 转换成 Token ID
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print("\n========== Token ID ==========")
print(input_ids)

# 3. 查看每个 Token
print("\n========== Token 明细 ==========")

for index, token_id in enumerate(input_ids):
    token = tokenizer.convert_ids_to_tokens(
        token_id
    )

    print(
        f"位置 {index:02d} | "
        f"ID {token_id:>8} | "
        f"Token {repr(token)}"
    )

# 4. 还原文本
decoded = tokenizer.decode(
    input_ids,
    skip_special_tokens=False
)

print("\n========== Decode ==========")
print(decoded)


========== 格式化文本 ==========
<|im_start|>system
你是一个数学助手。<|im_end|>
<|im_start|>user
2 + 3 等于多少？<|im_end|>
<|im_start|>assistant
5。<|im_end|>
<|im_start|>user
再加 4 呢？<|im_end|>
<|im_start|>assistant
9。<|im_end|>


========== Token ID ==========
{'input_ids': [151644, 8948, 198, 56568, 101909, 104552, 110498, 1773, 151645, 198, 151644, 872, 198, 17, 488, 220, 18, 10236, 255, 231, 34204, 100430, 11319, 151645, 198, 151644, 77091, 198, 20, 1773, 151645, 198, 151644, 872, 198, 87256, 20929, 220, 19, 4891, 239, 95, 11319, 151645, 198, 151644, 77091, 198, 24, 1773, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

========== Token 明细 ==========


ValueError: invalid literal for int() with base 10: 'i'

In [8]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

messages = [
    {
        "role": "system",
        "content": "你是一个数学助手。"
    },
    {
        "role": "user",
        "content": "2 + 3 等于多少？"
    },
    {
        "role": "assistant",
        "content": "5。"
    },
    {
        "role": "user",
        "content": "再加 4 呢？"
    },
    {
        "role": "assistant",
        "content": "9。"
    }
]

input_ids = []
labels = []

# 每次只格式化一个消息，并与前缀比较
# 这样可以观察消息边界是否符合预期
for i, message in enumerate(messages):
    prefix_ids = tokenizer.apply_chat_template(
        message,
        tokenzie=True,
        add_generation_prompt=False
    ) if i > 0 else []

    current_ids = tokenizer.apply_chat_template(
        messages[:i + 1],
        tokenize=True,
        add_generation_prompt=False
    )

    # 要求前面的完整消息不被模板重写
    if current_ids[:len(prefix_ids)] != prefix_ids:
        raise ValueError(
            "当前 Chat Template 不满足前缀稳定性，"
            "不能使用这种简单的差分方法。"
        )

    new_ids = current_ids[len(prefix_ids):]

    input_ids.extend(new_ids)

    if message["role"] == "assistant":
        # 教学简化：整个 Assistant 消息新增部分参与 Loss
        labels.extend(new_ids)
    else:
        labels.extend(
            [-100] * len(new_ids)
        )

input_ids = torch.tensor(
    [input_ids],
    dtype=torch.long
)

labels = torch.tensor(
    [labels],
    dtype=torch.long
)

attention_mask = torch.ones_like(
    input_ids
)

print("Input shape:", input_ids.shape)
print("Labels shape:", labels.shape)

print("\n========== 监督位置 ==========")

for token_id, label in zip(
    input_ids[0].tolist(),
    labels[0].tolist()
):
    token = tokenizer.convert_ids_to_tokens(
        token_id
    )

    print(
        f"{repr(token):25} "
        f"label={label}"
    )

# 只做一次前向传播，观察 Loss
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("\nLoss:", outputs.loss.item())



model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ValueError: 当前 Chat Template 不满足前缀稳定性，不能使用这种简单的差分方法。

In [9]:

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from trl import (
    SFTTrainer,
    SFTConfig
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

dataset = Dataset.from_list([
    {
        "messages": [
            {
                "role": "system",
                "content": "你是一个数学助手。"
            },
            {
                "role": "user",
                "content": "2 + 3 等于多少？"
            },
            {
                "role": "assistant",
                "content": "5。"
            },
            {
                "role": "user",
                "content": "再加 4 呢？"
            },
            {
                "role": "assistant",
                "content": "9。"
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "10 - 4 等于多少？"
            },
            {
                "role": "assistant",
                "content": "6。"
            }
        ]
    }
])

args = SFTConfig(
    output_dir="./multi_turn_sft",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=2,

    num_train_epochs=3,

    learning_rate=2e-5,

    max_length=256,

    # 不依赖模板生成区域标记
    assistant_only_loss=False,

    packing=False,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

trainer.save_model("./multi_turn_model")
tokenizer.save_pretrained("./multi_turn_model")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 169.19 MiB is free. Process 33332 has 1.50 GiB memory in use. Including non-PyTorch memory, this process has 3.98 GiB memory in use. Of the allocated memory 3.61 GiB is allocated by PyTorch, and 223.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [10]:

import random
import torch

from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

# ==========================================
# 1. 固定随机种子
# ==========================================

seed = 42

random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ==========================================
# 2. 加载模型和 Tokenizer
# ==========================================

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# Qwen 通常已定义 pad_token
assert tokenizer.pad_token_id is not None

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

# ==========================================
# 3. 准备训练数据
# ==========================================

system_prompt = (
    "你是颜色编码助手。"
    "只输出颜色代码，不要解释。"
    "红色对应 COLOR=RED，"
    "蓝色对应 COLOR=BLUE，"
    "绿色对应 COLOR=GREEN。"
)

pairs = [
    ("颜色：红色", "COLOR=RED"),
    ("颜色：蓝色", "COLOR=BLUE"),
    ("颜色：绿色", "COLOR=GREEN"),

    ("请编码红色", "COLOR=RED"),
    ("请编码蓝色", "COLOR=BLUE"),
    ("请编码绿色", "COLOR=GREEN"),

    ("把红色转换成代码", "COLOR=RED"),
    ("把蓝色转换成代码", "COLOR=BLUE"),
    ("把绿色转换成代码", "COLOR=GREEN"),

    ("输入颜色是红色", "COLOR=RED"),
    ("输入颜色是蓝色", "COLOR=BLUE"),
    ("输入颜色是绿色", "COLOR=GREEN")
]

samples = []

for question, answer in pairs:
    samples.append({
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]
    })

# ==========================================
# 4. Dataset
# ==========================================

class SFTDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.items = []

        for sample in samples:
            messages = sample["messages"]

            # 不包含最后一条 Assistant 回复
            prompt_messages = messages[:-1]

            # 推理时使用的 Prompt
            prompt_ids = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=True,
                add_generation_prompt=True
            )

            # 完整对话
            full_ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False
            )

            # 检查 Prompt 是否是完整序列的前缀
            if full_ids[:len(prompt_ids)] != prompt_ids:
                raise ValueError(
                    "Chat Template 前缀不匹配，"
                    "不能安全构造 Assistant-only Labels"
                )

            # Assistant 回复及结束标记
            answer_ids = full_ids[len(prompt_ids):]

            if not answer_ids:
                raise ValueError("Assistant 回复为空")

            labels = (
                [-100] * len(prompt_ids)
                + answer_ids
            )

            self.items.append({
                "input_ids": full_ids,
                "labels": labels
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


dataset = SFTDataset(
    samples,
    tokenizer
)

# ==========================================
# 5. Data Collator
# ==========================================

def collate_fn(batch):
    input_ids = [
        torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )
        for item in batch
    ]

    labels = [
        torch.tensor(
            item["labels"],
            dtype=torch.long
        )
        for item in batch
    ]

    attention_masks = [
        torch.ones_like(ids)
        for ids in input_ids
    ]

    # 右侧 Padding
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_mask = pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# ==========================================
# 6. 检查 Labels
# ==========================================

item = dataset[0]

print("\n========== Token 监督检查 ==========")

for token_id, label in zip(
    item["input_ids"],
    item["labels"]
):
    token = tokenizer.convert_ids_to_tokens(
        token_id
    )

    status = (
        "IGNORE"
        if label == -100
        else "TRAIN"
    )

    print(
        f"{repr(token):25} "
        f"label={label:8} "
        f"{status}"
    )

# ==========================================
# 7. 微调前的推理
# ==========================================

def generate_answer(model, question):
    model.eval()

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    model_device = next(model.parameters()).device

    inputs = {
        k: v.to(model_device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # 只解码新生成的 Token
    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


test_question = "颜色：红色"

base_model = base_model.to(device)

before_answer = generate_answer(
    base_model,
    test_question
)

print("\nBefore Training:", before_answer)

# ==========================================
# 8. 添加 LoRA
# ==========================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "v_proj"
    ],

    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    base_model,
    lora_config
)

model.print_trainable_parameters()

# ==========================================
# 9. TrainingArguments
# ==========================================

args = TrainingArguments(
    output_dir="./qwen_sft_lora_results",

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    num_train_epochs=10,

    learning_rate=2e-4,

    logging_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    report_to="none",

    seed=seed,

    remove_unused_columns=False
)

# ==========================================
# 10. 创建 Trainer
# ==========================================

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collate_fn
)

# ==========================================
# 11. 训练
# ==========================================

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

trainer.train()

# ==========================================
# 12. 显存统计
# ==========================================

if torch.cuda.is_available():
    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print(
        f"Peak GPU Memory: {peak_memory:.2f} GiB"
    )

# ==========================================
# 13. 微调后推理
# ==========================================

after_answer = generate_answer(
    model,
    test_question
)

print("\nBefore:", before_answer)
print("After :", after_answer)

# ==========================================
# 14. 保存 LoRA Adapter
# ==========================================

adapter_dir = "./my_qwen_lora_adapter"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Adapter saved:", adapter_dir)

# ==========================================
# 15. 重新加载 LoRA Adapter
# ==========================================

del trainer
del model
del base_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

reloaded_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
).to(device)

reloaded_model = PeftModel.from_pretrained(
    reloaded_base,
    adapter_dir
)

reloaded_model.eval()

print(
    "\nReloaded:",
    generate_answer(
        reloaded_model,
        test_question
    )
)


Device: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

ValueError: Chat Template 前缀不匹配，不能安全构造 Assistant-only Labels